# SRE-Bench GRPO Training in Google Colab

This notebook trains a language model to resolve production incidents using SRE-Bench environment and GRPO (Group Relative Policy Optimization).

## Prerequisites
- Google Account (for Colab and Drive)
- HuggingFace Account (for model access)
- Deployed SRE-Bench Space: `https://madhubuilds-sre-bench.hf.space`

## What We'll Do
1. Install dependencies
2. Clone the SRE-Bench repo
3. Connect to the deployed environment
4. Load Unsloth Qwen2.5-3B model
5. Run GRPO training with curriculum learning
6. Save trained model to Google Drive

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install unsloth "trl[peft]" openenv-core transformers datasets accelerate peft
!pip install -q -U google-colab

## Step 2: Clone SRE-Bench Repository

In [ ]:
# Clone the SRE-Bench repository
# Replace with your actual repo URL if different
!git clone https://github.com/YOUR_USERNAME/sre-bench.git /content/sre-bench
%cd /content/sre-bench
!pip install -e .

## Step 3: Load Model with Unsloth

In [ ]:
import os
import torch
from unsloth import FastLanguageModel, PatchDPOTrainer
from unsloth import is_bfloat16_supported

# Patch TRL for Unsloth optimizations
PatchDPOTrainer()

# Configuration
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 4096
LORA_RANK = 16

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"LoRA adapters added with rank={LORA_RANK}")

## Step 4: Run GRPO Training

This cell runs the GRPO training loop with curriculum learning. It connects to your deployed SRE-Bench Space.

In [ ]:
import re
import json
from collections import defaultdict
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

# Import SRE-Bench client
import sys
sys.path.insert(0, '/content/sre-bench')
from client import SreBenchEnv
from models import SreBenchAction

# Configuration
ENV_URL = "https://madhubuilds-sre-bench.hf.space"  # Your deployed Space
TRAIN_MAX_STEPS = 50
NUM_GENERATIONS = 4
TOTAL_CURRICULUM_ROLLOUTS = TRAIN_MAX_STEPS * NUM_GENERATIONS

CURRICULUM_STAGES = (
    ("easy", 0.30),
    ("medium", 0.70),
    ("hard", 1.00),
)

# Global state for training
global_model = model
global_tokenizer = tokenizer
global_rollout_counter = 0
global_last_difficulty = None
global_metric_sums = defaultdict(float)
global_metric_counts = defaultdict(int)
global_outcome_counts = defaultdict(int)

METRICS_LOG_PATH = "/content/sre_bench_grpo_outputs/rollout_metrics.jsonl"

# Ensure output directory exists
os.makedirs("/content/sre_bench_grpo_outputs", exist_ok=True)

# System prompt
SYSTEM_PROMPT = """[CRITICAL INSTRUCTION]
You are an autonomous Senior SRE Agent. You are NOT an assistant responding to a user. DO NOT write essays, tutorials, greetings, or general advice. Your ONLY purpose is to call tools to fix the cluster.

The cluster has 7 services: frontend, auth-api, order-service, payment-gateway, database, redis-queue, load-balancer.

Format your response EXACTLY like this:
<thought>
[Brief 1-sentence analysis of the current state]
</thought>
<tool>
{"tool_name": "grep_logs", "arguments": {"service": "database", "pattern": "ERROR"}, "hypothesis": "Checking DB for errors"}
</tool>

When you believe you have fixed the issue, use the resolve_incident tool:
<tool>
{"tool_name": "resolve_incident", "arguments": {"root_cause": "memory_leak", "fix_applied": "restarted frontend"}, "hypothesis": "fixed"}
</tool>

Any text outside of <thought> and <tool> tags is strictly forbidden.
"""

def parse_action_from_text(text: str) -> SreBenchAction:
    """Extracts JSON tool call from LLM generation."""
    match = re.search(r"<tool>(.*?)</tool>", text, re.DOTALL)
    if not match:
        return SreBenchAction(
            tool_name="grep_logs", 
            arguments={"service": "unknown", "pattern": ""},
            hypothesis="FAILED TO PARSE TOOL XML"
        )
    try:
        data = json.loads(match.group(1).strip())
        return SreBenchAction(
            tool_name=data.get("tool_name", "grep_logs"),
            arguments=data.get("arguments", {}),
            hypothesis=data.get("hypothesis", "")
        )
    except json.JSONDecodeError:
        return SreBenchAction(
            tool_name="grep_logs", 
            arguments={"service": "unknown", "pattern": ""},
            hypothesis="JSON DECODE ERROR"
        )

def get_curriculum_difficulty(rollout_index: int) -> str:
    """Return curriculum difficulty for a given rollout index."""
    progress = (rollout_index + 1) / max(1, TOTAL_CURRICULUM_ROLLOUTS)
    for difficulty, threshold in CURRICULUM_STAGES:
        if progress <= threshold:
            return difficulty
    return "hard"

def generate_trajectory(prompt_text: str, difficulty: str = "hard") -> dict:
    """Rolls out a full episode in SRE-Bench using the model."""
    global global_model, global_tokenizer
    
    with SreBenchEnv(base_url=ENV_URL).sync() as env:
        result = env.reset(difficulty=difficulty)
        obs = result.observation
        
        chat_history = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": obs.tool_output}
        ]
        
        done = False
        step = 0
        total_reward = 0.0
        
        while not done and step < 20:
            inputs = global_tokenizer.apply_chat_template(
                chat_history, 
                tokenize=True, 
                add_generation_prompt=True, 
                return_tensors="pt"
            ).to("cuda")
            
            with torch.no_grad():
                outputs = global_model.generate(
                    inputs, 
                    max_new_tokens=256, 
                    temperature=0.7,
                    pad_token_id=global_tokenizer.eos_token_id,
                    do_sample=True,
                )
            
            response = global_tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
            chat_history.append({"role": "assistant", "content": response})
            
            action = parse_action_from_text(response)
            result = env.step(action)
            obs = result.observation
            done = result.done
            total_reward = result.reward
            
            chat_history.append({"role": "user", "content": obs.tool_output})
            step += 1
            
        resolved = "=== INCIDENT RESOLVED ===" in obs.tool_output
        timed_out = "STEP BUDGET EXHAUSTED" in obs.tool_output
        successful = bool(obs.scores and obs.scores.get("root_cause_accuracy", 0.0) >= 1.0 and not timed_out)

        if hasattr(obs, 'scores') and obs.scores:
            return {
                "reward": total_reward,
                "scores": obs.scores,
                "difficulty": difficulty,
                "resolved": resolved,
                "timed_out": timed_out,
                "successful": successful,
                "steps": step,
            }
        else:
            return {
                "reward": total_reward,
                "scores": {},
                "difficulty": difficulty,
                "resolved": resolved,
                "timed_out": timed_out,
                "successful": successful,
                "steps": step,
            }

def format_reward_func(prompts, completions, **kwargs):
    """
    Rewards the model for correctly using thought and tool XML tags.
    This is a lightweight process-aware signal (Guide Step 9) that runs
    without hitting the environment, giving gradient signal even when
    env_reward_func rollouts are noisy.
    """
    rewards = []
    for comp in completions:
        text = comp[0] if isinstance(comp, list) else str(comp)
        has_tool_open  = "<tool>"  in text
        has_tool_close = "</tool>" in text
        has_thought    = "<thought>" in text
        # Full format: both thought and tool tags present
        if has_thought and has_tool_open and has_tool_close:
            rewards.append(0.3)
        # Partial: at least has tool tags
        elif has_tool_open and has_tool_close:
            rewards.append(0.15)
        # Nothing structured - penalise
        else:
            rewards.append(-0.1)
    return rewards

def env_reward_func(prompts, completions, **kwargs):
    """GRPO reward function that evaluates model completions."""
    global global_rollout_counter, global_last_difficulty
    global global_metric_sums, global_metric_counts, global_outcome_counts
    
    rewards = []
    for i, (prompt, comp) in enumerate(zip(prompts, completions)):
        rollout_index = global_rollout_counter
        difficulty = get_curriculum_difficulty(rollout_index)
        
        if difficulty != global_last_difficulty:
            print(f"[Curriculum] Switching to {difficulty.upper()} at rollout {rollout_index + 1}")
            global_last_difficulty = difficulty

        global_rollout_counter += 1
        result = generate_trajectory(prompt, difficulty=difficulty)
        
        r = result["reward"]
        scores = result["scores"]
        resolved = result["resolved"]
        timed_out = result["timed_out"]
        successful = result["successful"]
        steps = result["steps"]
        rewards.append(r)
        
        # Log metrics
        metric_row = {
            "rollout": rollout_index + 1,
            "difficulty": difficulty,
            "reward": r,
            "scores": scores,
            "resolved": resolved,
            "timed_out": timed_out,
            "successful": successful,
            "steps": steps,
        }
        with open(METRICS_LOG_PATH, "a") as f:
            f.write(json.dumps(metric_row) + "\n")

        global_metric_sums["reward"] += r
        global_metric_counts["reward"] += 1
        for score_name, score_value in scores.items():
            global_metric_sums[score_name] += float(score_value)
            global_metric_counts[score_name] += 1
        global_outcome_counts["episodes"] += 1
        global_outcome_counts["resolved"] += int(resolved)
        global_outcome_counts["timed_out"] += int(timed_out)
        global_outcome_counts["successful"] += int(successful)
        global_metric_sums["steps"] += float(steps)
        global_metric_counts["steps"] += 1
        
        if (rollout_index + 1) % 10 == 0:
            avg_reward = global_metric_sums["reward"] / max(1, global_metric_counts["reward"])
            print(f"[Metrics] Rollout {rollout_index + 1} | Reward={r:.4f} | Avg={avg_reward:.4f}")
    
    return rewards

# Create dummy dataset for GRPO
dummy_prompts = ["Resolve this incident"] * NUM_GENERATIONS
dataset = Dataset.from_list([{"prompt": p} for p in dummy_prompts])

# Configure GRPO
grpo_config = GRPOConfig(
    output_dir="/content/sre_bench_grpo_outputs",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    report_to="none",
    logging_steps=1,
    save_strategy="steps",
    save_steps=20,
    max_prompt_length=512,
    max_completion_length=256,  # Short limit forces <tool> JSON output, prevents essay hallucinations
)

# Initialize trainer
trainer = GRPOTrainer(
    model=model,
    train_dataset=dataset,
    reward_funcs=[env_reward_func, format_reward_func],  # Multi-signal: env outcome + format compliance (Guide Step 7)
    args=grpo_config,
    tokenizer=tokenizer,
)

# Train!
print("Starting GRPO training...")
trainer.train()
print("Training complete!")

## Step 5: Save Trained Model to Google Drive

In [ ]:
# Mount Google Drive and save model
from google.colab import drive
drive.mount('/content/drive')

# Save the model
output_dir = "/content/drive/MyDrive/sre_bench_trained_model"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model saved to: {output_dir}")

# Also copy the metrics log
import shutil
shutil.copy(METRICS_LOG_PATH, f"/content/drive/MyDrive/sre_bench_trained_model/rollout_metrics.jsonl")
print("Metrics log saved to Google Drive!")

## Next Steps

After training completes:
1. Download the trained model from Google Drive
2. Run the evaluation script to compare baseline vs trained
3. Generate PNG curves from the metrics log
4. Commit all outputs to your repo

Run this locally:
```bash
# Download model from Drive, then:
python training/evaluate_baseline_vs_trained.py \
  --env_url https://madhubuilds-sre-bench.hf.space \
  --episodes 25 \
  --model_path ./sre_bench_trained_model
```